In [15]:
# Merge shadow effects into the main final dataset


import geopandas as gpd
import numpy as np

print("=== Creating dataset with Shadow ===")

# 1. Load the main economic dataset
main = gpd.read_file("accra_buildings_solar_roi_final.gpkg")
print(f"Main dataset: {len(main):,} buildings")

# 2. Load the shadow dataset
shadow = gpd.read_file("accra_buildings_with_shadow_v5.gpkg") 
print(f"Shadow dataset: {len(shadow):,} buildings")

# 3. Merge shadow columns into main
buildings = main.merge(
    shadow[['building_id', 'height_m', 'weighted_shadow_m', 'solar_adjusted_kwh']],
    on='building_id',
    how='left'
)
print("Merge completed.")

# 4. Calculate base solar potential
irradiance = 1779
efficiency = 0.205
performance_ratio = 0.76

buildings["solar_kwh_year"] = (
    buildings["usable_area_m2"]
    * irradiance
    * efficiency
    * performance_ratio
)

# 5. Use shadow-adjusted value (or fallback)
buildings["solar_adjusted_kwh_final"] = buildings["solar_adjusted_kwh"].fillna(
    buildings["solar_kwh_year"] * 0.95
)

# Clip extremes
buildings["solar_adjusted_kwh_final"] = buildings["solar_adjusted_kwh_final"].clip(upper=500_000)

print("\nShadow integration done.")
print(buildings[["solar_kwh_year", "solar_adjusted_kwh_final"]].describe().round(0))

=== Creating dataset with Shadow ===
Main dataset: 632,195 buildings
Shadow dataset: 632,195 buildings
Merge completed.

Shadow integration done.
       solar_kwh_year  solar_adjusted_kwh_final
count        630946.0                  632195.0
mean          14896.0                   12408.0
std           28801.0                   16998.0
min             311.0                     323.0
25%            3241.0                    2954.0
50%            7013.0                    6261.0
75%           18516.0                   16147.0
max         5503902.0                  328722.0


In [20]:
# Polish - Better Index and Class

print("=== Final Polish ===")

# Load the current complete dataset
buildings = gpd.read_file("accra_buildings_solar_roi_final_complete.gpkg")

# Balanced Solar Index (60% energy output + 40% roof suitability)
max_solar = buildings["solar_adjusted_kwh_final"].max()

buildings["solar_index"] = (
    0.60 * (np.log1p(buildings["solar_adjusted_kwh_final"]) / np.log1p(max_solar)) * 100 +
    0.40 * buildings["suitability_score"]
)

# Data-driven classification
q75 = buildings["solar_index"].quantile(0.75)
q50 = buildings["solar_index"].quantile(0.50)
q25 = buildings["solar_index"].quantile(0.25)

conditions = [
    buildings["solar_index"] > q75,
    buildings["solar_index"] > q50,
    buildings["solar_index"] > q25
]
labels = ["Very High", "High", "Moderate"]
buildings["solar_class"] = np.select(conditions, labels, default="Low")

print("\nUltimate Final Summary:")
print(buildings[["solar_kwh_year", "solar_adjusted_kwh_final", "solar_index"]].describe().round(1))

print(f"\nSolar Class Distribution (based on solar_index):\n{buildings['solar_class'].value_counts()}")

# Policy and investor friendly
buildings["co2_savings_tonnes"] = (buildings["solar_adjusted_kwh_final"] * 0.4) / 1000   # kg - tonnes
buildings["system_kw"] = buildings["usable_area_m2"] * 0.20   # rough panel density

print("\nAdded CO₂ savings and system size:")
print(buildings[["co2_savings_tonnes", "system_kw"]].describe().round(2))

# Save
ultimate_file = "accra_buildings_solar_roi_final_complete.gpkg"
buildings.to_file(ultimate_file, driver="GPKG")

print(f"\nUltimate final dataset saved as: {ultimate_file}")
print("Project core is now complete and ready for dashboard or GitHub.")

=== Final Polish ===

Ultimate Final Summary:
       solar_kwh_year  solar_adjusted_kwh_final  solar_index
count        630946.0                  632195.0     630946.0
mean          14896.0                   12408.0         63.1
std           28801.2                   16997.9          9.0
min             310.8                     322.6         38.7
25%            3240.7                    2954.0         56.1
50%            7012.7                    6261.2         62.3
75%           18515.7                   16147.4         69.3
max         5503902.1                  328722.5         98.3

Solar Class Distribution (based on solar_index):
solar_class
Low          158986
Very High    157737
High         157736
Moderate     157736
Name: count, dtype: int64

Added CO₂ savings and system size:
       co2_savings_tonnes  system_kw
count           632195.00  630946.00
mean                 4.96      10.75
std                  6.80      20.78
min                  0.13       0.22
25%             

In [22]:
import geopandas as gpd

print("Converting to Parquet for faster loading...")

gdf = gpd.read_file("accra_buildings_solar_roi_final_complete.gpkg")

gdf.to_parquet("accra_buildings_solar_roi_final_complete.parquet")

print("Parquet file created.")

Converting to Parquet for faster loading...
Conversion done! Parquet file created.
